# Single Index Model: A Deeper Dive
This notebook expands on the Single Index Model (SIM) material introduced in the Session 1 lecture. We develop the mathematical details behind SIM parameter estimation, bootstrap uncertainty quantification, the SIM-derived covariance matrix, and the Second-Order Cone Program (SOCP) formulation for maximum Sharpe ratio portfolios. No code is included here; see the companion example notebooks for implementation.
___

## The SIM Regression
The Single Index Model posits that the continuously compounded growth rate of asset $i$ at time $t$ is a linear function of the market growth rate $g_M(t)$:
$$
g_i(t) = \alpha_i + \beta_i\,g_M(t) + \varepsilon_i(t)
$$
where:
* $\alpha_i$ is the **intercept** (asset $i$'s return independent of the market),
* $\beta_i$ is the **market sensitivity** (how much asset $i$ moves per unit market move),
* $g_M(t)$ is the continuously compounded growth rate of the market index at time $t$,
* $\varepsilon_i(t) \sim \mathcal{N}(0, \Delta t\,\sigma_{\varepsilon,i}^2)$ is a zero-mean idiosyncratic noise term.

The key assumptions are that the noise terms $\varepsilon_i(t)$ are uncorrelated across assets ($\text{Cov}(\varepsilon_i, \varepsilon_j) = 0$ for $i \neq j$) and independent of the market return ($\text{Cov}(\varepsilon_i, g_M) = 0$). These assumptions give the SIM its power: all pairwise asset correlations flow through the single market factor.
___

## Estimating SIM Parameters via Regularized OLS
Given $T$ observations of market returns $g_M(t)$ and asset returns $g_i(t)$, we estimate the parameter vector $\boldsymbol{\theta}_i = (\alpha_i, \beta_i)^{\top}$ by solving a regularized least-squares problem.

> __Why regularization?__
>
> When the number of observations $T$ is small relative to the condition number of $\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}$, ordinary least squares can produce unstable estimates. Adding a Ridge penalty $\delta \geq 0$ shrinks the estimates toward zero and stabilizes the solution. Setting $\delta = 0$ recovers standard OLS.

Define the $T \times 2$ design matrix $\hat{\mathbf{X}} = [\mathbf{1} \;\; \mathbf{g}_M]$, where $\mathbf{1}$ is the $T$-vector of ones and $\mathbf{g}_M = (g_M(1), \ldots, g_M(T))^{\top}$ is the vector of market returns. Let $\mathbf{y}_i = (g_i(1), \ldots, g_i(T))^{\top}$ be the vector of asset $i$'s returns. The regularized OLS problem is:
$$
\hat{\boldsymbol{\theta}}_i = \arg\min_{\boldsymbol{\theta}} \left[\frac{1}{2}\|\mathbf{y}_i - \hat{\mathbf{X}}\boldsymbol{\theta}\|_2^2 + \frac{\delta}{2}\|\boldsymbol{\theta}\|_2^2\right]
$$
Taking the gradient with respect to $\boldsymbol{\theta}$, setting it to zero, and solving gives the closed-form solution:
$$
\boxed{\hat{\boldsymbol{\theta}}_i = \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\,\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y}_i}
$$

Once we have $\hat{\boldsymbol{\theta}}_i = (\hat{\alpha}_i, \hat{\beta}_i)^{\top}$, we compute the residuals $\hat{\boldsymbol{\varepsilon}}_i = \mathbf{y}_i - \hat{\mathbf{X}}\hat{\boldsymbol{\theta}}_i$ and estimate the idiosyncratic volatility as:
$$
\hat{\sigma}_{\varepsilon,i} = \sqrt{\frac{1}{\Delta t(T - p)}\|\mathbf{y}_i - \hat{\mathbf{X}}\hat{\boldsymbol{\theta}}_i\|_2^2}
$$
where $p = 2$ is the number of estimated parameters ($\alpha_i$ and $\beta_i$), and the $T - p$ denominator is the degrees-of-freedom correction.
___

## Bootstrap Uncertainty Quantification
Point estimates of $(\hat{\alpha}_i, \hat{\beta}_i)$ are useful, but how certain are we? The bootstrap generates an empirical sampling distribution by repeatedly creating synthetic datasets and re-estimating the parameters.

> __Parametric bootstrap:__
>
> We use a parametric bootstrap (rather than a nonparametric resample) because the SIM has a well-specified error model. We simulate new noise from the fitted residual distribution, add it to the fitted signal, and re-estimate. This directly probes how estimation uncertainty propagates through the regression.

The procedure is as follows. For each bootstrap replicate $k = 1, \ldots, K$:

1. **Sample synthetic errors**: $\boldsymbol{\varepsilon}^{(k)} \sim \mathcal{N}(\mathbf{0},\, \Delta t\,\hat{\sigma}_{\varepsilon}^2\,\mathbf{I})$
2. **Create synthetic observations**: $\mathbf{y}^{(k)} = \hat{\mathbf{X}}\hat{\boldsymbol{\theta}} + \boldsymbol{\varepsilon}^{(k)}$
3. **Re-estimate**: $\hat{\boldsymbol{\theta}}^{(k)} = \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\,\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y}^{(k)}$

After $K$ replicates, the collection $\{\hat{\boldsymbol{\theta}}^{(1)}, \ldots, \hat{\boldsymbol{\theta}}^{(K)}\}$ approximates the sampling distribution of $\hat{\boldsymbol{\theta}}$.

> __What does the bootstrap converge to?__
>
> Because both the data generation and estimation steps are linear in the noise, we can derive the limiting distribution analytically. Substituting the synthetic data into the estimator gives $\hat{\boldsymbol{\theta}}^{(k)} = \mathbf{S}\boldsymbol{\theta} + \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\,\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\boldsymbol{\varepsilon}^{(k)}$, where $\boldsymbol{\varepsilon}^{(k)}$ is Gaussian.

The empirical distribution converges to:
$$
\boxed{\hat{\boldsymbol{\theta}} \sim \mathcal{N}\!\left(\mathbf{S}\boldsymbol{\theta},\; \Delta t\,\hat{\sigma}_{\varepsilon}^2\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\,\mathbf{I}\right)^{-1}\right)}
$$
where $\mathbf{S} = \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}} + \delta\,\mathbf{I}\right)^{-1}\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}$ is the **shrinkage matrix**. When $\delta = 0$, we have $\mathbf{S} = \mathbf{I}$ and the estimator is unbiased. When $\delta > 0$, the estimator is biased toward zero, but the variance is reduced. This is the classic bias-variance tradeoff of Ridge regression.
___

## The SIM-Derived Covariance Matrix
One of the main reasons to use the SIM is to obtain a structured covariance matrix with far fewer parameters than a full sample covariance. For a universe of $N$ assets, the full covariance matrix has $N(N+1)/2$ free parameters, which can be enormous. The SIM covariance requires only $2N + 1$ parameters: $N$ betas, $N$ idiosyncratic variances, and one market variance.

> __Parameter reduction:__
>
> Consider $N = 500$ assets. The full covariance has $125{,}250$ free parameters. The SIM covariance has $1{,}001$. This dramatic reduction stabilizes portfolio optimization, especially when $T$ is not much larger than $N$.

To derive the SIM covariance, start from the model $g_i(t) = \alpha_i + \beta_i\,g_M(t) + \varepsilon_i(t)$. For assets $i$ and $j$:
$$
\text{Cov}(g_i, g_j) = \text{Cov}\bigl(\beta_i\,g_M + \varepsilon_i,\; \beta_j\,g_M + \varepsilon_j\bigr)
$$
Expanding and using the assumptions that $\varepsilon_i$ is independent of $g_M$ and that $\text{Cov}(\varepsilon_i, \varepsilon_j) = 0$ for $i \neq j$:
$$
\text{Cov}(g_i, g_j) = \beta_i\,\beta_j\,\text{Var}(g_M) + \text{Cov}(\varepsilon_i, \varepsilon_j)
$$
The cross-covariance of the noise terms vanishes when $i \neq j$, and equals $\Delta t\,\sigma_{\varepsilon,i}^2$ when $i = j$. Letting $\sigma_M^2 = \text{Var}(g_M)$, we arrive at:
$$
\boxed{\Sigma_{ij}^{\text{SIM}} = \begin{cases} \beta_i^2\,\sigma_M^2 + \Delta t\,\sigma_{\varepsilon,i}^2 & \text{if } i = j \\ \beta_i\,\beta_j\,\sigma_M^2 & \text{if } i \neq j \end{cases}}
$$

In matrix form, this can be written compactly as $\boldsymbol{\Sigma}^{\text{SIM}} = \sigma_M^2\,\boldsymbol{\beta}\boldsymbol{\beta}^{\top} + \Delta t\,\text{diag}(\sigma_{\varepsilon,1}^2, \ldots, \sigma_{\varepsilon,N}^2)$, where $\boldsymbol{\beta} = (\beta_1, \ldots, \beta_N)^{\top}$. The first term is a rank-1 matrix capturing systematic risk, and the second is a diagonal matrix capturing idiosyncratic risk.
___

## SOCP Formulation for Maximum Sharpe Ratio
The Sharpe ratio of a portfolio with weight vector $\mathbf{w}$ is defined as:
$$
\text{SR}(\mathbf{w}) = \frac{\mathbb{E}[g_{\mathcal{P}}] - g_f}{\sigma_{\mathcal{P}}} = \frac{\mathbf{c}^{\top}\mathbf{w}}{\sqrt{\mathbf{w}^{\top}\boldsymbol{\Sigma}\mathbf{w}}}
$$
where $\mathbf{c} = \boldsymbol{\alpha} + \boldsymbol{\beta}\,g_M - g_f\,\mathbf{1}$ is the vector of expected excess returns and $g_f$ is the risk-free rate. Because this is a ratio (not a linear or quadratic objective), it cannot be solved directly as a standard QP.

> __Why SOCP?__
>
> Maximizing a ratio is equivalent to maximizing the numerator while constraining the denominator. This reformulation yields a second-order cone constraint, which modern interior-point solvers handle efficiently. The solver [Clarabel.jl](https://github.com/oxfordcontrol/Clarabel.jl) is well-suited for this class of problems.

To reformulate, compute the Cholesky decomposition $\boldsymbol{\Sigma} = \mathbf{U}^{\top}\mathbf{U}$ so that $\sigma_{\mathcal{P}} = \|\mathbf{U}\mathbf{w}\|_2$. The maximum Sharpe ratio problem becomes:
$$
\max_{\mathbf{w}} \; \mathbf{c}^{\top}\mathbf{w}
$$
subject to:
$$
\boxed{\begin{bmatrix} \mathbf{c}^{\top}\mathbf{w}\,/\,\tau \\ \mathbf{U}\mathbf{w} \end{bmatrix} \in \mathcal{K}_{\text{SOC}}, \quad \sum_{i=1}^{N} w_i = 1, \quad w_i \geq 0}
$$
where $\tau > 0$ is a target Sharpe ratio parameter, and $\mathcal{K}_{\text{SOC}} = \{(s, \mathbf{z}) : s \geq \|\mathbf{z}\|_2\}$ is the second-order cone. The cone constraint enforces $\mathbf{c}^{\top}\mathbf{w}/\tau \geq \|\mathbf{U}\mathbf{w}\|_2$, which is equivalent to requiring $\text{SR}(\mathbf{w}) \geq \tau$. By maximizing the objective while satisfying this constraint, the solver finds the portfolio with the highest achievable Sharpe ratio.

The long-only constraint $w_i \geq 0$ and the budget constraint $\sum w_i = 1$ are standard linear constraints. Together with the cone constraint, the full problem is a Second-Order Cone Program, solvable in polynomial time.